<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.8-monitoring-and-observability/notebooks/exercises-10_8.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.8: Monitoring & Observability

*Module 10 · AI System Architecture & Production Deployment*

> Your AI gateway is deployed, scaling, and serving traffic. But: Is latency spiking? Are tokens burning faster than budget? Is one model throwing more errors? You need dashboards, metrics, and alerts — purpose-built for AI workloads. This lesson builds the complete observability stack.

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-10.8.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Structured Logging — Every Request Tells a Story — `part1_logging.py`
2. Step 2 — Custom Metrics — The 6 Numbers That Matter — `part2_metrics.py`
3. Step 3 — Alerting Policies — Get Notified Before Users Complain — `part3_alerts.py`
4. Step 4 — ObservabilityStack — One Middleware, Complete Visibility — `part4_middleware.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q fastapi uvicorn requests


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Structured Logging — Every Request Tells a Story

JSON logs that Cloud Logging, Datadog, and Grafana can parse, filter, and alert on.

**`part1_logging.py`** · _Block 1 of 4_

STRUCTURED LOGGING FOR AI APPS — JSON logs with AI-specific fields.


In [ ]:
# =============================================
# STRUCTURED LOGGING FOR AI APPS
# JSON logs with AI-specific fields.
# Compatible with Cloud Logging severity levels.
# =============================================

import json, time, uuid, os, sys
from datetime import datetime
from contextvars import ContextVar

# Request-scoped trace ID
_request_id: ContextVar[str] = ContextVar("request_id", default="")

class AILogger:
    """Structured JSON logger for AI applications."""
    
    # Cloud Logging severity levels
    DEBUG = "DEBUG"
    INFO = "INFO"
    WARNING = "WARNING"
    ERROR = "ERROR"
    CRITICAL = "CRITICAL"
    
    def __init__(self, service: str = "ai-gateway", version: str = ""):
        self.service = service
        self.version = version or os.getenv("VERSION", "unknown")
    
    def _emit(self, severity: str, message: str, **fields):
        """Emit a structured JSON log entry."""
        entry = {
            "severity": severity,
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "message": message,
            "service": self.service,
            "version": self.version,
        }
        
        # Add request trace ID if available
        req_id = _request_id.get("")
        if req_id:
            entry["request_id"] = req_id
        
        # Merge any extra fields
        entry.update(fields)
        
        # Print as single JSON line (Cloud Logging auto-parses)
        print(json.dumps(entry), flush=True)
    
    def llm_request(self, *, model: str, team: str, input_tokens: int,
                     output_tokens: int, latency_ms: float, cost_usd: float,
                     cached: bool = False, status: str = "success", error: str = ""):
        """Log a complete LLM request with all AI-specific fields."""
        severity = self.INFO if status == "success" else self.ERROR
        self._emit(severity, f"LLM {status}: {model}",
            event_type="llm_request",
            model=model, team=team,
            input_tokens=input_tokens, output_tokens=output_tokens,
            total_tokens=input_tokens + output_tokens,
            latency_ms=round(latency_ms, 1),
            cost_usd=round(cost_usd, 6),
            cached=cached, status=status,
            error=error[:200] if error else "",
        )
    
    def cache_event(self, hit: bool, cache_type: str, latency_ms: float):
        """Log a cache hit/miss."""
        self._emit(self.DEBUG, f"Cache {'hit' if hit else 'miss'}: {cache_type}",
            event_type="cache", cache_hit=hit, cache_type=cache_type,
            latency_ms=round(latency_ms, 1))
    
    def alert(self, alert_type: str, message: str, **fields):
        """Log a warning/alert condition."""
        self._emit(self.WARNING, message, event_type="alert",
                   alert_type=alert_type, **fields)

# ── Demo ──
logger = AILogger(service="netsetos-gateway", version="a1b2c3d")

_request_id.set("req-8f3a2b")

logger.llm_request(model="gemini-2.0-flash", team="pro_team",
    input_tokens=1500, output_tokens=400, latency_ms=845, cost_usd=0.00031)
logger.cache_event(hit=True, cache_type="semantic", latency_ms=8.2)
logger.alert("budget_warning", "Team pro_team at 85% daily budget", team="pro_team", spent=42.5, limit=50.0)


> **What just happened?** AILogger emits single-line JSON logs that Cloud Logging auto-parses. Three log types: llm_request (model, team, tokens, latency, cost, status), cache (hit/miss, type, lookup latency), and alert (budget warnings, error spikes). A ContextVar traces request_id across all logs in one request. These structured fields enable Cloud Logging queries like: jsonPayload.event_type="llm_request" AND jsonPayload.latency_ms>5000 — find all slow LLM calls instantly.


### Step 2 · Custom Metrics — The 6 Numbers That Matter

Latency, tokens, cost, errors, cache hits, and quality — tracked per model and per team.

**`part2_metrics.py`** · _Block 2 of 4_

CUSTOM METRICS FOR AI APPLICATIONS — 6 metric types: latency, tokens, cost,


In [ ]:
# =============================================
# CUSTOM METRICS FOR AI APPLICATIONS
# 6 metric types: latency, tokens, cost,
# errors, cache, quality.
# Compatible with Prometheus / Cloud Monitoring.
# =============================================

import time
from collections import defaultdict
from dataclasses import dataclass, field

class AIMetrics:
    """Track AI-specific metrics with labels (model, team)."""
    
    def __init__(self):
        # Counters (monotonically increasing)
        self.request_count = defaultdict(int)      # by model, team, status
        self.token_count = defaultdict(int)        # by model, team, direction (in/out)
        self.cost_total = defaultdict(float)       # by model, team
        self.cache_hits = defaultdict(int)         # by cache_type
        self.cache_misses = defaultdict(int)       # by cache_type
        
        # Histograms (distribution of values)
        self.latency_buckets = defaultdict(list)  # by model
        
        # Gauges (current value)
        self.active_requests = defaultdict(int)   # by model
        self.daily_spend = defaultdict(float)     # by team
    
    def record_request(self, model: str, team: str, status: str,
                       input_tokens: int, output_tokens: int,
                       latency_ms: float, cost_usd: float):
        """Record all metrics for one LLM request."""
        # Request count
        self.request_count[f"{model}|{team}|{status}"] += 1
        
        # Token count
        self.token_count[f"{model}|{team}|input"] += input_tokens
        self.token_count[f"{model}|{team}|output"] += output_tokens
        
        # Cost
        self.cost_total[f"{model}|{team}"] += cost_usd
        self.daily_spend[team] += cost_usd
        
        # Latency histogram
        self.latency_buckets[model].append(latency_ms)
        if len(self.latency_buckets[model]) > 1000:
            self.latency_buckets[model] = self.latency_buckets[model][-500:]
    
    def record_cache(self, hit: bool, cache_type: str):
        """Record a cache hit or miss."""
        if hit:
            self.cache_hits[cache_type] += 1
        else:
            self.cache_misses[cache_type] += 1
    
    def get_latency_percentiles(self, model: str) -> dict:
        """Calculate p50, p95, p99 latency for a model."""
        values = sorted(self.latency_buckets.get(model, []))
        if not values:
            return {"p50": 0, "p95": 0, "p99": 0}
        def pct(p): return values[int(len(values) * p / 100)]
        return {"p50": round(pct(50)), "p95": round(pct(95)), "p99": round(pct(99))}
    
    def get_error_rate(self, model: str) -> float:
        """Calculate error rate for a model."""
        success = sum(v for k, v in self.request_count.items()
                      if k.startswith(model) and k.endswith("success"))
        errors = sum(v for k, v in self.request_count.items()
                     if k.startswith(model) and k.endswith("error"))
        total = success + errors
        return errors / total if total > 0 else 0
    
    def get_cache_hit_rate(self) -> float:
        """Overall cache hit rate."""
        hits = sum(self.cache_hits.values())
        misses = sum(self.cache_misses.values())
        return hits / (hits + misses) if (hits + misses) > 0 else 0
    
    def dashboard(self):
        """Print a real-time metrics dashboard."""
        total_req = sum(self.request_count.values())
        total_cost = sum(self.cost_total.values())
        total_tokens = sum(self.token_count.values())
        
        print(f"\n  ╔══════════════════════════════════════╗")
        print(f"  ║   AI GATEWAY — LIVE DASHBOARD        ║")
        print(f"  ╠══════════════════════════════════════╣")
        print(f"  ║  Requests:   {total_req:>8,}               ║")
        print(f"  ║  Tokens:     {total_tokens:>8,}               ║")
        print(f"  ║  Cost:       ${total_cost:>8.4f}               ║")
        print(f"  ║  Cache rate: {self.get_cache_hit_rate():>8.0%}               ║")
        print(f"  ╠══════════════════════════════════════╣")
        
        # Per-model latency
        for model in self.latency_buckets:
            pct = self.get_latency_percentiles(model)
            err = self.get_error_rate(model)
            print(f"  ║  {model[:20]:20s}              ║")
            print(f"  ║    p50={pct['p50']:>5}ms  p95={pct['p95']:>5}ms       ║")
            print(f"  ║    p99={pct['p99']:>5}ms  err={err:>5.1%}       ║")
        
        print(f"  ╚══════════════════════════════════════╝")

# ── Demo ──
metrics = AIMetrics()
import random

# Simulate 100 requests
for _ in range(80):
    metrics.record_request("gemini-2.0-flash", "pro_team", "success",
        1500, 400, random.gauss(800, 200), 0.0003)
    metrics.record_cache(random.random() < 0.4, "exact")

for _ in range(15):
    metrics.record_request("gemini-2.5-pro", "pro_team", "success",
        5000, 2000, random.gauss(3000, 800), 0.026)

for _ in range(5):
    metrics.record_request("gemini-2.0-flash", "pro_team", "error",
        1500, 0, 5000, 0)

metrics.dashboard()


> **What just happened?** AIMetrics tracks 6 metric types with labels: request count (by model/team/status), token count (input vs output), cost (running total + daily spend by team), latency (histogram with p50/p95/p99 percentiles), error rate (per model), cache hit rate (overall and by type). The dashboard renders a live ASCII view. The p95 latency tells you what 95% of users experience. The p99 catches the tail — if p99 is 10x the p50, you have a long-tail problem.


### Step 3 · Alerting Policies — Get Notified Before Users Complain

Automated alerts for the 5 things that break AI apps in production.

**`part3_alerts.py`** · _Block 3 of 4_

ALERTING POLICIES FOR AI APPLICATIONS — 5 alert types with configurable thresholds.


In [ ]:
# =============================================
# ALERTING POLICIES FOR AI APPLICATIONS
# 5 alert types with configurable thresholds.
# =============================================

from dataclasses import dataclass
from enum import Enum

class Severity(Enum):
    INFO = "info"
    WARNING = "warning"
    CRITICAL = "critical"

@dataclass
class AlertPolicy:
    name: str
    condition: str
    threshold: float
    severity: Severity
    notification: str   # "slack", "email", "pager"

class AlertManager:
    """Evaluate alert policies against current metrics."""
    
    def __init__(self, metrics: AIMetrics, logger: AILogger):
        self.metrics = metrics
        self.logger = logger
        self.fired: list[dict] = []
        
        # The 5 alerts every AI app needs
        self.policies = [
            AlertPolicy(
                "latency_p95",
                "p95 latency exceeds threshold",
                threshold=5000,  # 5 seconds
                severity=Severity.WARNING,
                notification="slack",
            ),
            AlertPolicy(
                "error_rate",
                "Error rate exceeds threshold",
                threshold=0.05,  # 5%
                severity=Severity.CRITICAL,
                notification="pager",
            ),
            AlertPolicy(
                "daily_budget",
                "Daily spend exceeds 80% of budget",
                threshold=0.80,  # 80% of budget
                severity=Severity.WARNING,
                notification="email",
            ),
            AlertPolicy(
                "cache_hit_rate",
                "Cache hit rate dropped below threshold",
                threshold=0.20,  # below 20% = caching broken
                severity=Severity.WARNING,
                notification="slack",
            ),
            AlertPolicy(
                "token_spike",
                "Average tokens per request spiking",
                threshold=5000,  # avg tokens > 5K = prompt bloat
                severity=Severity.INFO,
                notification="slack",
            ),
        ]
    
    def evaluate(self, team: str = "", budget: float = 50.0):
        """Check all policies against current metrics."""
        print("Alert evaluation:\n")
        
        for policy in self.policies:
            triggered = False
            value = 0
            
            if policy.name == "latency_p95":
                for model in self.metrics.latency_buckets:
                    pct = self.metrics.get_latency_percentiles(model)
                    value = pct["p95"]
                    if value > policy.threshold:
                        triggered = True
            
            elif policy.name == "error_rate":
                for model in self.metrics.latency_buckets:
                    value = self.metrics.get_error_rate(model)
                    if value > policy.threshold:
                        triggered = True
            
            elif policy.name == "daily_budget":
                spend = self.metrics.daily_spend.get(team, 0)
                value = spend / budget if budget > 0 else 0
                triggered = value > policy.threshold
            
            elif policy.name == "cache_hit_rate":
                value = self.metrics.get_cache_hit_rate()
                triggered = value < policy.threshold and sum(self.metrics.cache_hits.values()) + sum(self.metrics.cache_misses.values()) > 10
            
            elif policy.name == "token_spike":
                total_tokens = sum(self.metrics.token_count.values())
                total_reqs = sum(self.metrics.request_count.values())
                value = total_tokens / total_reqs if total_reqs > 0 else 0
                triggered = value > policy.threshold
            
            icon = "🔴" if triggered and policy.severity == Severity.CRITICAL \
                   else "🟡" if triggered else "🟢"
            status = "TRIGGERED" if triggered else "ok"
            print(f"  {icon} {policy.name:20s} value={value:>8.2f}  threshold={policy.threshold}  → {status}")
            
            if triggered:
                self.fired.append({"policy": policy.name, "severity": policy.severity.value,
                                   "value": value, "notify": policy.notification})
                self.logger.alert(policy.name, policy.condition, value=value, threshold=policy.threshold)

# ── Demo ──
alert_mgr = AlertManager(metrics, logger)
alert_mgr.evaluate(team="pro_team", budget=50.0)


> **What just happened?** AlertManager checks 5 policies against live metrics: latency_p95 > 5s → Slack warning (service degrading). Error rate > 5% → pager critical (something is broken). Daily spend > 80% → email warning (budget running out). Cache hit rate < 20% → Slack warning (caching broken, costs spiking). Avg tokens > 5K → Slack info (prompt bloat detected). Each alert fires to the right channel. You get notified before users complain, before the budget runs out, and before a broken cache doubles your API bill overnight.


### Step 4 · ObservabilityStack — One Middleware, Complete Visibility

Wrap your FastAPI app with logging + metrics + alerts in one middleware.

**`part4_middleware.py`** · _Block 4 of 4_

OBSERVABILITY MIDDLEWARE — Wraps every request with: logging, metrics,


In [ ]:
# =============================================
# OBSERVABILITY MIDDLEWARE
# Wraps every request with: logging, metrics,
# tracing, and alert evaluation.
# =============================================

from fastapi import FastAPI, Request, Response
from starlette.middleware.base import BaseHTTPMiddleware
import uuid, time

class ObservabilityMiddleware(BaseHTTPMiddleware):
    """Auto-instrument every request with logging + metrics."""
    
    def __init__(self, app, logger: AILogger, metrics: AIMetrics):
        super().__init__(app)
        self.logger = logger
        self.metrics = metrics
    
    async def dispatch(self, request: Request, call_next):
        # Generate request ID
        request_id = request.headers.get("X-Request-ID", f"req-{uuid.uuid4().hex[:8]}")
        _request_id.set(request_id)
        
        start = time.time()
        
        # Process request
        try:
            response = await call_next(request)
            latency_ms = (time.time() - start) * 1000
            
            # Add trace headers to response
            response.headers["X-Request-ID"] = request_id
            response.headers["X-Latency-Ms"] = str(round(latency_ms))
            
            # Log HTTP-level request
            self.logger._emit(AILogger.INFO, f"{request.method} {request.url.path}",
                event_type="http_request",
                method=request.method,
                path=str(request.url.path),
                status_code=response.status_code,
                latency_ms=round(latency_ms, 1))
            
            return response
            
        except Exception as e:
            latency_ms = (time.time() - start) * 1000
            self.logger._emit(AILogger.ERROR, f"Request failed: {e}",
                event_type="http_request",
                method=request.method,
                path=str(request.url.path),
                status_code=500,
                latency_ms=round(latency_ms, 1),
                error=str(e)[:200])
            raise

# ── Complete wired app ──
def create_monitored_app() -> FastAPI:
    """Create a FastAPI app with full observability."""
    app = FastAPI(title="Netsetos AI Gateway (Monitored)")
    
    logger = AILogger(service="ai-gateway")
    metrics = AIMetrics()
    alerts = AlertManager(metrics, logger)
    
    # Add observability middleware
    app.add_middleware(ObservabilityMiddleware, logger=logger, metrics=metrics)
    
    # Store in app state for access in endpoints
    app.state.logger = logger
    app.state.metrics = metrics
    app.state.alerts = alerts
    
    @app.get("/health")
    async def health():
        return {"status": "healthy"}
    
    @app.post("/v1/chat")
    async def chat(request: dict):
        model = request.get("model", "gemini-2.0-flash")
        start = time.time()
        
        # Your LLM call here...
        response_text = f"Response from {model}"
        latency_ms = (time.time() - start) * 1000
        
        # Record AI-specific metrics
        metrics.record_request(model, "demo_team", "success",
            1500, 400, latency_ms, 0.0003)
        
        # Log AI-specific details
        logger.llm_request(model=model, team="demo_team",
            input_tokens=1500, output_tokens=400,
            latency_ms=latency_ms, cost_usd=0.0003)
        
        return {"response": response_text, "model": model}
    
    # Metrics dashboard endpoint
    @app.get("/metrics")
    async def get_metrics():
        return {
            "requests": sum(metrics.request_count.values()),
            "tokens": sum(metrics.token_count.values()),
            "cost_usd": round(sum(metrics.cost_total.values()), 4),
            "cache_hit_rate": round(metrics.get_cache_hit_rate(), 3),
            "models": {
                model: {
                    **metrics.get_latency_percentiles(model),
                    "error_rate": round(metrics.get_error_rate(model), 3),
                }
                for model in metrics.latency_buckets
            },
        }
    
    return app

print("""
Usage:
  app = create_monitored_app()
  # Run with: uvicorn main:app --port 8080

  Every request automatically gets:
    ✅ Request ID tracing (X-Request-ID header)
    ✅ HTTP-level structured logging
    ✅ Latency tracking in response headers
    ✅ AI-specific metrics on /metrics endpoint
    ✅ Error logging with stack context
""")


> **What just happened?** ObservabilityMiddleware auto-instruments every request: assigns a trace ID ( X-Request-ID ), measures latency, logs HTTP details, and returns latency in response headers. The /metrics endpoint exposes live metrics in JSON: request count, total tokens, cost, cache hit rate, and per-model latency percentiles + error rates. create_monitored_app() wires everything together: middleware + logger + metrics + alerts. Add two lines to your existing app — add_middleware(ObservabilityMiddleware) — and you get complete observability.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
